# CV PDF Extraction with OpenAI

Notebook dung CV dang PDF, goi OpenAI API (key lay tu back-end/.env) va tra ket qua theo schema JSON.

In [12]:
# Chay cell nay neu kernel chua co thu vien can thiet
%pip install -q openai pymupdf python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [13]:
import json
import os
import random
import re
import time
import fitz
from datetime import date
from pathlib import Path
from typing import Any

from dotenv import load_dotenv
from openai import APIConnectionError, APIError, APITimeoutError, OpenAI, RateLimitError


def find_backend_env(start: Path) -> Path:
    for current in [start, *start.parents]:
        env_path = current / "back-end" / ".env"
        if env_path.exists():
            return env_path
    raise FileNotFoundError("Khong tim thay file back-end/.env")


start_path = Path.cwd().resolve()
env_path = find_backend_env(start_path)
load_dotenv(env_path)

api_key = (os.getenv("OPENAI_API_KEY") or "").strip().strip('"').strip("'")
if not api_key:
    raise ValueError("Khong tim thay OPENAI_API_KEY trong file .env")

model_name = (os.getenv("OPENAI_MODEL_NAME") or "gpt-4o-mini").strip().strip('"').strip("'")
fallback_models_env = (os.getenv("OPENAI_MODEL_FALLBACKS") or "").strip().strip('"').strip("'")
if fallback_models_env:
    fallback_models = [item.strip() for item in fallback_models_env.split(",") if item.strip()]
else:
    default_fallbacks = ["gpt-4.1-mini", "gpt-4o-mini"]
    fallback_models = [item for item in default_fallbacks if item != model_name]

client = OpenAI(api_key=api_key)

print(f"Da nap .env tu: {env_path}")
print(f"Dang su dung OpenAI model chinh: {model_name}")
if fallback_models:
    print(f"Model du phong: {', '.join(fallback_models)}")

Da nap .env tu: D:\datn\source_code\back-end\.env
Dang su dung OpenAI model chinh: gpt-4o-mini
Model du phong: gpt-4.1-mini


In [14]:
MONTH_NAME_TO_NUMBER = {
    "jan": 1,
    "january": 1,
    "feb": 2,
    "february": 2,
    "mar": 3,
    "march": 3,
    "apr": 4,
    "april": 4,
    "may": 5,
    "jun": 6,
    "june": 6,
    "jul": 7,
    "july": 7,
    "aug": 8,
    "august": 8,
    "sep": 9,
    "sept": 9,
    "september": 9,
    "oct": 10,
    "october": 10,
    "nov": 11,
    "november": 11,
    "dec": 12,
    "december": 12,
}

PRESENT_MARKERS = ("present", "current", "now", "hien tai", "den nay", "to date")


def read_pdf_text(pdf_path):
    doc = fitz.open(pdf_path)
    text = ""
    for page in doc:
        text += page.get_text()
    return text


def normalize_duration_text(duration: str) -> str:
    text = str(duration or "").lower().strip()
    text = text.replace("–", "-").replace("—", "-")
    text = re.sub(r"\b(to|den|toi|until)\b", "-", text)
    text = re.sub(r"\s+", " ", text)
    return text


def parse_date_token(token: str, prefer_end: bool = False) -> tuple[int, int] | None:
    cleaned = re.sub(r"[\(\)\[\],]", " ", token.lower())
    cleaned = re.sub(r"\s+", " ", cleaned).strip()
    if not cleaned:
        return None

    today = date.today()
    if any(marker in cleaned for marker in PRESENT_MARKERS):
        return (today.year, today.month)

    month_year_match = re.search(r"(?<!\d)(0?[1-9]|1[0-2])\s*[/-]\s*((?:19|20)\d{2})(?!\d)", cleaned)
    if month_year_match:
        month = int(month_year_match.group(1))
        year = int(month_year_match.group(2))
        return (year, month)

    year_month_match = re.search(r"(?<!\d)((?:19|20)\d{2})\s*[/-]\s*(0?[1-9]|1[0-2])(?!\d)", cleaned)
    if year_month_match:
        year = int(year_month_match.group(1))
        month = int(year_month_match.group(2))
        return (year, month)

    vietnamese_month_match = re.search(r"thang\s*(0?[1-9]|1[0-2])\s*((?:19|20)\d{2})", cleaned)
    if vietnamese_month_match:
        month = int(vietnamese_month_match.group(1))
        year = int(vietnamese_month_match.group(2))
        return (year, month)

    for month_name, month_number in MONTH_NAME_TO_NUMBER.items():
        if re.search(rf"\b{re.escape(month_name)}\b", cleaned):
            year_match = re.search(r"(?<!\d)((?:19|20)\d{2})(?!\d)", cleaned)
            if year_match:
                return (int(year_match.group(1)), month_number)

    year_match = re.search(r"(?<!\d)((?:19|20)\d{2})(?!\d)", cleaned)
    if year_match:
        month = 12 if prefer_end else 1
        return (int(year_match.group(1)), month)

    return None


def parse_duration_range_to_months(duration: str) -> tuple[int, int] | None:
    cleaned = normalize_duration_text(duration)
    if "-" not in cleaned:
        return None

    start_token, end_token = cleaned.split("-", 1)
    start_date = parse_date_token(start_token, prefer_end=False)
    end_date = parse_date_token(end_token, prefer_end=True)
    if not start_date or not end_date:
        return None

    start_index = (start_date[0] * 12) + (start_date[1] - 1)
    end_index = (end_date[0] * 12) + (end_date[1] - 1)
    if end_index < start_index:
        return None

    return (start_index, end_index)


def parse_duration_length_to_months(duration: str) -> int:
    cleaned = normalize_duration_text(duration)

    compact_match = re.search(
        r"(\d+(?:\.\d+)?)\s*y(?:ears?)?\s*(\d+(?:\.\d+)?)\s*m(?:onths?)?",
        cleaned,
    )
    if compact_match:
        years = float(compact_match.group(1))
        months = float(compact_match.group(2))
        return int(round((years * 12) + months))

    years = 0.0
    months = 0.0

    year_match = re.search(r"(\d+(?:\.\d+)?)\s*(?:\+\s*)?(?:years?|yrs?|yr|y|nam)", cleaned)
    if year_match:
        years = float(year_match.group(1))

    month_match = re.search(r"(\d+(?:\.\d+)?)\s*(?:months?|mos?|mo|m|thang)", cleaned)
    if month_match:
        months = float(month_match.group(1))

    total_months = int(round((years * 12) + months))
    return max(total_months, 0)


def merge_month_ranges(ranges: list[tuple[int, int]]) -> int:
    if not ranges:
        return 0

    sorted_ranges = sorted(ranges, key=lambda item: item[0])
    merged: list[list[int]] = [[sorted_ranges[0][0], sorted_ranges[0][1]]]

    for start, end in sorted_ranges[1:]:
        last_range = merged[-1]
        if start <= last_range[1] + 1:
            last_range[1] = max(last_range[1], end)
        else:
            merged.append([start, end])

    return sum((end - start + 1) for start, end in merged)


def calculate_experience_years(experience_items: list[dict[str, str]]) -> float:
    parsed_ranges: list[tuple[int, int]] = []
    additive_months = 0

    for item in experience_items:
        duration = str(item.get("duration", "") or "").strip()
        if not duration:
            continue

        range_months = parse_duration_range_to_months(duration)
        if range_months:
            parsed_ranges.append(range_months)
            continue

        additive_months += parse_duration_length_to_months(duration)

    total_months = merge_month_ranges(parsed_ranges) + additive_months
    if total_months <= 0:
        return 0.0

    return round(total_months / 12, 2)


def normalize_cv_result(data: dict[str, Any]) -> dict[str, Any]:
    # ===== EXPERIENCE =====
    experience_items = data.get("experience", [])
    if not isinstance(experience_items, list):
        experience_items = []

    normalized_experience = []
    for item in experience_items:
        if isinstance(item, dict):
            normalized_experience.append(
                {
                    "title": str(item.get("title", "") or ""),
                    "company": str(item.get("company", "") or ""),
                    "duration": str(item.get("duration", "") or ""),
                    "description": str(item.get("description", "") or ""),
                }
            )

    # ===== EDUCATION =====
    education_items = data.get("education", [])
    if not isinstance(education_items, list):
        education_items = []

    # ===== SKILLS =====
    skills = data.get("skills", {})
    if not isinstance(skills, dict):
        skills = {}

    core_skills = skills.get("core", [])
    soft_skills = skills.get("soft", [])

    if not isinstance(core_skills, list):
        core_skills = []
    if not isinstance(soft_skills, list):
        soft_skills = []

    core_skills = [str(s).lower().strip() for s in core_skills]
    soft_skills = [str(s).lower().strip() for s in soft_skills]

    # ==== CERTIFICATIONS =====
    certifications = data.get("certifications", [])
    if not isinstance(certifications, list):
        certifications = []
    certifications = [str(c).strip() for c in certifications if isinstance(c, str)]

    # ===== EXPERIENCE YEARS =====
    experience_years = calculate_experience_years(normalized_experience)
    if experience_years <= 0:
        fallback_experience_years = data.get("experience_years", 0)
        try:
            experience_years = float(fallback_experience_years)
        except (TypeError, ValueError):
            experience_years = 0.0

    return {
        "name": str(data.get("name", "") or ""),
        "self-description": str(data.get("self-description", "") or ""),
        "skills": {
            "core": list(set(core_skills)),
            "soft": list(set(soft_skills)),
        },
        "experience": normalized_experience,
        "education": [str(item) for item in education_items],
        "certifications": certifications,
        "experience_years": experience_years,
    }

In [15]:
RETRYABLE_STATUS_CODES = {408, 409, 429, 500, 502, 503, 504}


def is_retryable_openai_error(error: Exception) -> bool:
    if isinstance(error, (RateLimitError, APITimeoutError, APIConnectionError)):
        return True

    if isinstance(error, APIError):
        status_code = getattr(error, "status_code", None)
        if isinstance(status_code, int) and status_code in RETRYABLE_STATUS_CODES:
            return True

    text = str(error).upper()
    markers = ("RATE LIMIT", "TIMEOUT", "429", "500", "502", "503", "504")
    return any(marker in text for marker in markers)


def request_openai_json(prompt: str, model: str, max_retries: int = 5) -> str:
    last_error: Exception | None = None

    for attempt in range(1, max_retries + 1):
        try:
            response = client.chat.completions.create(
                model=model,
                temperature=0,
                response_format={"type": "json_object"},
                messages=[
                    {
                        "role": "system",
                        "content": "Ban la he thong trich xuat thong tin tu CV. Chi tra ve JSON hop le, khong them giai thich.",
                    },
                    {"role": "user", "content": prompt},
                ],
            )

            content = (response.choices[0].message.content or "").strip()
            if not content:
                raise ValueError("OpenAI tra ve noi dung rong")
            return content
        except Exception as error:
            last_error = error
            if (not is_retryable_openai_error(error)) or attempt == max_retries:
                raise
            wait_seconds = (2 ** (attempt - 1)) + random.uniform(0, 0.8)
            print(
                f"[OpenAI] Model {model} tam loi (lan {attempt}/{max_retries}), "
                f"thu lai sau {wait_seconds:.1f}s..."
            )
            time.sleep(wait_seconds)

    raise RuntimeError("Khong nhan duoc phan hoi tu OpenAI") from last_error


def clean_json_response(raw_content: str) -> str:
    raw_content = raw_content.strip()
    if raw_content.startswith("```"):
        raw_content = raw_content.strip("`")
        raw_content = raw_content.replace("json\n", "", 1).strip()
    return raw_content


def extract_cv_info(cv_text: str, model: str | None = None, max_retries: int = 5) -> dict[str, Any]:
    schema_description = {
        "name": "",
        "self-description": "",
        "skills": {
            "core": [""],
            "soft": [""],},
        "experience": [
            {
                "title": "",
                "company": "",
                "duration": "",
                "description": "",
            }
        ],
        "education": [""],
        "certifications": [""],
        "experience_years": 0,
    }

    prompt = (
        "Hãy đọc CV dưới đây và trích xuất thông tin theo đúng schema sau:\n"
        f"{json.dumps(schema_description, ensure_ascii=False)}\n\n"
        "Yêu cầu:\n"
        "- Đọc toàn bộ nội dung và trích xuất đầy đủ thông tin có thể.\n"
        "- Nếu thiếu thông tin thì để chuỗi rỗng \"\", mảng rỗng [] hoặc giá trị 0.\n"
        "- Trường experience_years phải là số.\n"
        "- Tổng hợp đầy đủ các kỹ năng, cần đọc và tìm cả những kỹ năng trong công việc được mô tả ở phần kinh nghiệm sau đó phân loại hợp lý.\n"
        "- skills.core: kỹ năng chuyên môn (technical/domain skills)\n"
        "- skills.soft: kỹ năng mềm (soft skills)\n"
        "- certifications: các chứng chỉ, bằng cấp, khóa học đã hoàn thành (nếu có)\n\n"
        "Nội dung CV:\n"
        f"{cv_text}"
    )

    model_candidates: list[str] = []
    if model:
        model_candidates.append(model)
    if model_name not in model_candidates:
        model_candidates.append(model_name)
    for fallback_model in fallback_models:
        if fallback_model not in model_candidates:
            model_candidates.append(fallback_model)

    last_error: Exception | None = None
    for model_to_use in model_candidates:
        try:
            raw_content = request_openai_json(
                prompt=prompt,
                model=model_to_use,
                max_retries=max_retries,
            )
            parsed = json.loads(clean_json_response(raw_content))
            return normalize_cv_result(parsed)
        except Exception as error:
            last_error = error
            print(f"[OpenAI] Goi model {model_to_use} khong thanh cong: {error}")

    raise RuntimeError(
        "Khong the trich xuat CV sau khi da retry va thu model du phong. "
        "Hay kiem tra OPENAI_API_KEY, OPENAI_MODEL_NAME hoac thu lai sau."
    ) from last_error

In [16]:
# Doi duong dan nay thanh file CV ban muon xu ly
pdf_path = Path(r"D:\datn\source_code\ai-service\data\CV\0403 Bùi Lan Hương Anh - IT Comtor - thaontt2.pdf")

cv_text = read_pdf_text(pdf_path)
# print(cv_text)
cv_info = extract_cv_info(cv_text, max_retries=5)

workspace_root = env_path.parent.parent
cv_output_path = workspace_root / "ai-service" / "notebooks" / "cv.json"
cv_output_path.write_text(
    json.dumps(cv_info, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print(f"Saved JSON to: {cv_output_path}")
print(json.dumps(cv_info, ensure_ascii=False, indent=2))

Saved JSON to: D:\datn\source_code\ai-service\notebooks\cv.json
{
  "name": "BÙI LAN HƯƠNG ANH",
  "self-description": "Với một tinh thần nhiệt huyết và ham học hỏi, tôi luôn cố gắng, nỗ lực hoàn thành tốt những công việc được giao. Tốt nghiệp chuyên ngành ngôn ngữ Hàn cùng kinh nghiệm hơn 3 năm làm việc tại các công ty Hàn Quốc và Việt Nam, tôi tin rằng sẽ mình đảm nhiệm tốt công việc tại quý công ty, tiến xa hơn trong công việc.",
  "skills": {
    "core": [
      "biên dịch tài liệu kỹ thuật",
      "phiên dịch tiếng hàn - việt, việt - hàn",
      "tin học văn phòng"
    ],
    "soft": [
      "kỹ năng giải quyết vấn đề",
      "kỹ năng làm việc nhóm",
      "kỹ năng giao tiếp"
    ]
  },
  "experience": [
    {
      "title": "Phiên dịch tiếng Hàn",
      "company": "Công ty TNHH Samsung Electronics VN Thái Nguyên",
      "duration": "Tháng 06/2022 - tháng 03/2024",
      "description": "Phiên dịch tiếng Hàn - Việt, Việt - Hàn trong bộ phận sản xuất. Biên dịch tài liệu kỹ thuật chu